# Use case: keep a folder of notes searchable and clean

A notes folder on disk (Obsidian, Logseq, plain markdown) with `[[wikilinks]]`. You want to
re-index it after every edit without re-embedding everything, find the notes you wrote twice,
and browse what goes with what. This notebook writes a small vault into a temp folder and runs
that housekeeping loop:

1. index the folder; edit one file; re-index and see that only the change is embedded
2. `find_duplicates()` to catch two notes that say the same thing
3. `related()` blends similarity with the links you wrote
4. `forget()` a note and confirm it is gone from search and from the graph

**Needs:** Ollama with `nomic-embed-text` and the `[graph]` extra (`pip install slim-llm-memory[graph]`).
The last step also needs `qwen2.5:7b-instruct`.

In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
STORE = ROOT / ".usecase_nb" / "notes"          # the store lives here; delete the folder to start over
RUN_LLM = True                               # the steps that call a chat model are slow on CPU
LLM = "qwen2.5:7b-instruct"                      # follows context better than llama3.2:3b

import shutil
from slim_llm_memory import topic

VAULT = ROOT / ".usecase_nb" / "notes-vault"
shutil.rmtree(VAULT, ignore_errors=True); VAULT.mkdir(parents=True)
shutil.rmtree(STORE, ignore_errors=True)

NOTES = {
    "sourdough.md":   "# Sourdough\nFeed the [[starter]] 8 hours before mixing. Bulk ferment 5 hours at 24 °C, "
                      "shape, then cold proof overnight. Bake at 250 °C in a dutch oven, lid on for 20 minutes.",
    "starter.md":     "# Starter\nEqual weights flour and water, fed daily. A ripe starter doubles in 4 to 6 hours "
                      "and smells sour but not like acetone. See [[sourdough]] for the bake.",
    "pizza.md":       "# Pizza dough\n65 % hydration, a pinch of yeast, 24 hours in the fridge. Uses the same "
                      "shaping as [[sourdough]] but no starter.",
    "espresso.md":    "# Espresso\n18 g in, 36 g out, 27 seconds. Grind finer if it runs fast. "
                      "Beans from [[roaster-notes]].",
    "roaster-notes.md": "# Roaster notes\nThe Ethiopian from the Saturday market roaster is best 10 to 20 days "
                        "after roast. Light roasts need a hotter water temperature, about 94 °C.",
    "bread-timing.md": "# Bread timing\nFeed the starter about eight hours before you mix. Bulk ferment around five "
                       "hours at 24 degrees, shape, and cold proof in the fridge overnight. Bake at 250 with the "
                       "lid on for the first twenty minutes.",
}
for name, text in NOTES.items():
    (VAULT / name).write_text(text)

notes = topic("notes", path=STORE)
print(notes.add(VAULT))
print(sorted(notes.graph.edges()))

added 6 doc(s), 6 chunks: 6 embedded, 0 unchanged, 0 removed
[('espresso.md', 'roaster-notes.md', 'links', 1.0), ('pizza.md', 'sourdough.md', 'links', 1.0), ('sourdough.md', 'starter.md', 'links', 1.0), ('starter.md', 'sourdough.md', 'links', 1.0)]


Six notes, seven chunks, and the `[[wikilinks]]` became `links` edges without any extra call.

## 1. Edit one file, re-index everything

Append a line to one note and run `add(VAULT)` again over the whole folder. Only the changed
chunk is embedded; everything else reports `unchanged`.

In [2]:
(VAULT / "espresso.md").write_text(NOTES["espresso.md"] + "\nFor milk drinks, pull a ristretto: 18 g in, 27 g out.")
print(notes.add(VAULT))
print(notes.ask("ratio for a ristretto", k=1).top.text)

added 6 doc(s), 6 chunks: 1 embedded, 5 unchanged, 0 removed
# Espresso

18 g in, 36 g out, 27 seconds. Grind finer if it runs fast. Beans from [[roaster-notes]].
For milk drinks, pull a ristretto: 18 g in, 27 g out.


## 2. Notes you wrote twice

`bread-timing.md` is `sourdough.md` in other words. `find_duplicates()` clusters chunks whose
cosine is above a threshold; anything it returns is worth a look before merging.

In [3]:
for h in notes.memory.neighbours("sourdough.md#0", k=2):
    print(f"{h.score:.2f}  {h.id}")
print("clusters:", notes.memory.find_duplicates(threshold=0.86))

0.87  bread-timing.md#0
0.85  starter.md#0
clusters: [['bread-timing.md#0', 'sourdough.md#0']]


## 3. What goes with this note?

`related()` scores other notes by 0.6 × cosine of their chunks plus 0.4 × link weight. A note
you linked ranks up even when its text is different; a note that merely reads alike ranks by
similarity alone. `via` tells you which.

In [4]:
print(notes.related("sourdough.md", k=4))

ask("related('sourdough.md')")  4 hit(s) · related · embed 0 ms · scan 0.34 ms
   1  0.85  starter.md#0             # Starter Equal weights flour and water, fed daily. A ripe sta  [graph]
   2  0.76  pizza.md#0               # Pizza dough 65 % hydration, a pinch of yeast, 24 hours in th  [graph]
   3  0.87  bread-timing.md#0        # Bread timing Feed the starter about eight hours before you m  [vector]
   4  0.66  roaster-notes.md#0       # Roaster notes The Ethiopian from the Saturday market roaster  [vector]


In [5]:
print(notes.related("espresso.md", k=3))

ask("related('espresso.md')")  3 hit(s) · related · embed 0 ms · scan 0.15 ms
   1  0.66  roaster-notes.md#0       # Roaster notes The Ethiopian from the Saturday market roaster  [graph]
   2  0.63  bread-timing.md#0        # Bread timing Feed the starter about eight hours before you m  [vector]
   3  0.61  sourdough.md#0           # Sourdough Feed the [[starter]] 8 hours before mixing. Bulk f  [vector]


## 4. Forget a note

Merge the duplicate into `sourdough.md` and drop it. `forget()` removes its chunks and its graph
node; the store is saved at once.

In [6]:
print("removed chunks:", notes.forget("bread-timing.md"))
print("docs now:", notes.docs())
print("duplicates now:", notes.memory.find_duplicates(threshold=0.86))

removed chunks:

 1
docs now: ['espresso.md', 'pizza.md', 'roaster-notes.md', 'sourdough.md', 'starter.md']
duplicates now: []


## 5. Entities, if you have time

`add(enrich=True)` runs a local model over each **new or changed** chunk and stores the names it
finds in `meta["entities"]`, plus `mentions` edges in the graph. Then `ask(entity="...")` keeps
only chunks about that one thing. It is one model call per chunk, so it goes into a separate
fresh store here and is gated by `RUN_LLM`.

In [7]:
if RUN_LLM:
    enriched = topic("notes enriched", path=STORE.parent / "notes-enriched")
    enriched.add(VAULT, enrich=LLM)               # fresh store, so every chunk is new and gets enriched
    ents = enriched.entities()
    print(ents)
    ent = "Ethiopian" if "Ethiopian" in ents else next(iter(ents))
    print(f"\nonly chunks about {ent!r}:")
    print(enriched.ask("what temperature and timing?", k=3, min_score=0.0, entity=ent))
    enriched.close()

{'sourdough': 2, 'starter': 2, '# Sourdough': 1, '27 seconds': 1, '36 g': 1, '[[roaster-notes]]': 1, 'dutch oven': 1, 'Espresso': 1, 'Ethiopian': 1, 'five hours': 1, 'flour': 1, 'Grinder': 1, 'lid': 1, 'Pizza dough': 1, 'Ristretto': 1, 'roast': 1, 'Saturday market': 1, 'Starter': 1, 'twenty minutes': 1, 'water': 1, 'yeast': 1}

only chunks about 'Ethiopian':


ask('what temperature and timing?')  1 hit(s) · hybrid · embed 810 ms · scan 1.21 ms
   1  0.58  roaster-notes.md#0       # Roaster notes The Ethiopian from the Saturday market roaster  [both]


## Takeaways

- **Re-index the folder on every change.** Content hashing makes it cost one embedding per edited chunk.
- **Duplicates are a query, not a cleanup project.** Run `find_duplicates()` weekly; merge, `forget()`.
- **Links you write are retrieval signal.** `[[wikilinks]]` need no extra step; `related()` uses them.
- **The store is next to the vault, not inside it.** Delete the store folder and rebuild any time.

In [8]:
notes.close()